# Threshold Tradeoffs and Equalized Odds Lab


## Purpose

This lab explores how decision thresholds change fairness metrics.

Learning goals:

- Sweep thresholds.
- Compare selection-rate gaps and error-rate gaps.
- Interpret fairness as a tradeoff rather than a single optimum.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 3000

audit = pd.DataFrame({
    "group": rng.choice(["A", "B"], size=n, p=[0.55, 0.45]),
    "target": rng.binomial(1, 0.40, size=n),
})

audit["score"] = np.where(
    audit["target"] == 1,
    rng.beta(5, 3, size=n),
    rng.beta(3, 5, size=n),
)

audit.loc[audit["group"] == "B", "score"] = np.clip(
    audit.loc[audit["group"] == "B", "score"] - 0.04,
    0,
    1,
)

def summarize_at_threshold(frame, threshold):
    temp = frame.copy()
    temp["prediction"] = (temp["score"] >= threshold).astype(int)

    rows = []

    for group_name, data in temp.groupby("group"):
        tp = ((data["prediction"] == 1) & (data["target"] == 1)).sum()
        fp = ((data["prediction"] == 1) & (data["target"] == 0)).sum()
        fn = ((data["prediction"] == 0) & (data["target"] == 1)).sum()
        tn = ((data["prediction"] == 0) & (data["target"] == 0)).sum()

        rows.append({
            "group": group_name,
            "selection_rate": data["prediction"].mean(),
            "tpr": tp / max(tp + fn, 1),
            "fpr": fp / max(fp + tn, 1),
        })

    metrics = pd.DataFrame(rows)

    return {
        "threshold": threshold,
        "selection_gap": metrics["selection_rate"].max() - metrics["selection_rate"].min(),
        "tpr_gap": metrics["tpr"].max() - metrics["tpr"].min(),
        "fpr_gap": metrics["fpr"].max() - metrics["fpr"].min(),
    }

threshold_table = pd.DataFrame([
    summarize_at_threshold(audit, threshold)
    for threshold in np.linspace(0.10, 0.90, 17)
])

threshold_table

## Interpretation

Changing a threshold can improve one fairness metric while worsening another. Threshold policy should be approved and documented.